# Tutorial 1.2 — Part 2: Introduction to day-ahead load forecasting

This tutorial complements the [data-quality and imputation tutorial](tutorial_1_2_time_series_forecasting.ipynb) (each notebook is independent). Here we use the original, complete 2007 GEFCom2012 aggregate demand series.

The aim is to practise the steps used in the assessment: construct predictors, split data in time order, fit a model, generate forecasts and interpret errors. Work through one method at a time. Use ordinary notebook cells for forecasting. Construct reusable functions for error metrics.

By the end, you should be able to:

- issue a forecast for the next 24 hours and move the origin forward by one day;
- implement persistence and daily/weekly seasonal-naïve forecasts;
- construct and visualise load-lag and calendar predictors for linear regression;
- calculate RMSE, MAE and MAPE and compare forecasts fairly.

Allow approximately 75–90 minutes: setup/origins 10 minutes, benchmarks 20 minutes, linear features/model 30 minutes, metrics/plots 20 minutes and discussion 5 minutes. Replace the **TODOs** and remove their NotImplementedError once each step is complete.

## 1. Setup and data

Use Z21, the aggregate load across 20 zones, measured in MW. We restrict this tutorial to the complete year 2007. Loading and checking the data are supplied so you can focus on forecasting.

Source: Hong, T., Pinson, P. and Fan, S. (2014), ‘Global Energy Forecasting Competition 2012’, *International Journal of Forecasting*, 30(2), pp. 357–363. [Dataset paper](https://doi.org/10.1016/j.ijforecast.2013.07.001). Further provenance is in [the data README](data/tutorial_1_2/README.md).

Run from the repository root. No other notebook or local Python helper is needed.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

In [ ]:
DATA_PATH = Path("data") / "tutorial_1_2" / "load_data_full.csv"
load_data = pd.read_csv(DATA_PATH, index_col=0, parse_dates=[0]).sort_index()
load_data.index.name = "timestamp"
if not load_data.index.is_unique:
    raise ValueError("Resolve duplicate timestamps before constructing load lags.")

load_series = load_data.loc["2007", "Z21"].copy()
expected_index = pd.date_range("2007-01-01 00:00", "2007-12-31 23:00", freq="h", name="timestamp")
if not load_series.index.equals(expected_index):
    raise ValueError("The selected 2007 series must contain every hourly timestamp.")
if not np.isfinite(load_series.to_numpy()).all() or (load_series <= 0).any():
    raise ValueError("This exercise requires complete, finite, positive demand observations.")
print(f"Loaded {len(load_series):,} complete hourly observations from 2007.")

## 2. Forecast origins and chronological split

At **23:00 on day $D-1$**, issue forecasts for **00:00–23:00 on day $D$**. These are 24 forecasts from the same origin, with lead times 1–24 hours. Move the origin forward **24 hours**, observe the completed day, then forecast the next day.

Training: **1 January–30 September 2007**. Testing: **1 October–31 December 2007**. We forecast the entire test set and use one week for the final illustration. Fit linear regression once on training data; keep its coefficients fixed while updating the available lagged observations each day.

![Daily forecast origins](images/tutorial_1_2/rolling_day_ahead_schematic_v2.png)

The figure above visualises the forecast setting. Each square represents one day. Blue days have been observed; orange is the next day's forecast. Each row visualises how the forecast origin advances.

Note that this is a simplified end-of-day forecasting exercise. Typically, a forecast made earlier in the preceding day would have less recent demand information available.

In [ ]:
TRAIN_START, TRAIN_END = "2007-01-01 00:00", "2007-09-30 23:00"
TEST_START, TEST_END = "2007-10-01 00:00", "2007-12-31 23:00"
PLOT_START, PLOT_END = "2007-10-08 00:00", "2007-10-14 23:00"

training_load = load_series.loc[TRAIN_START:TRAIN_END]
test_actual = load_series.loc[TEST_START:TEST_END]
test_index = test_actual.index
forecast_origins = pd.Series(
    test_index.normalize() - pd.Timedelta(hours=1), index=test_index, name="origin"
)
forecasts = pd.DataFrame(index=test_index)

assert len(test_actual) == 2208 and len(test_actual) % 24 == 0
assert forecast_origins.iloc[0] == training_load.index[-1]
print(f"Training observations: {len(training_load):,}")
print(f"Test observations: {len(test_actual):,}, covering {len(test_actual) // 24} daily origins.")
print(f"First origin: {forecast_origins.iloc[0]}; next origin: {forecast_origins.iloc[24]}")

In [ ]:
# Supplied exploratory plot: use training observations only.
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
training_load.loc["2007-09-17":TRAIN_END].plot(ax=axes[0], color="tab:blue")
axes[0].set(title="Two weeks of observed training demand", xlabel="Timestamp", ylabel="Demand (MW)")
for label, selected in [("Weekdays", training_load.index.dayofweek < 5),
                        ("Weekends", training_load.index.dayofweek >= 5)]:
    group = training_load.loc[selected]
    group.groupby(group.index.hour).mean().plot(ax=axes[1], marker="o", markersize=3, label=label)
axes[1].set(title="Mean training demand by hour", xlabel="Hour of day", ylabel="Mean demand (MW)")
axes[1].set_xticks(range(0, 24, 3))
axes[1].legend()
fig.tight_layout()
plt.show()

## 3. Persistence (naïve forecast)

### Task 1

Repeat the demand observed at 23:00 across all 24 hours of the following day:

$$\hat y_{D,h}=y_{D-1,23},\qquad h=0,\ldots,23.$$

Use the short loop below to move the origin forward one day at a time. Each day therefore has a flat forecast, but its level can change at the next origin.

Do **not** use the previous observation at every test hour: that would be a one-hour-ahead forecast, not this day-ahead experiment.

In [ ]:
naive_predictions = pd.Series(index=test_index, dtype=float)
for day_start in test_index[::24]:
    origin = day_start - pd.Timedelta(hours=1)
    day_index = pd.date_range(day_start, periods=24, freq="h")
    # TODO: assign the demand observed at origin to every timestamp in day_index.
    raise NotImplementedError("Complete the daily persistence forecast.")
forecasts["Naïve"] = naive_predictions

In [ ]:
# Supplied plot: the first two daily forecasts.
fig, ax = plt.subplots(figsize=(11, 3))
test_actual.iloc[:48].plot(ax=ax, color="black", label="Actual")
forecasts["Naïve"].iloc[:48].plot(ax=ax, label="Naïve")
ax.set(title="Persistence: a new flat forecast each day", xlabel="Timestamp", ylabel="Demand (MW)")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Daily seasonal-naïve forecast

### Task 2

For each target hour, copy demand from the **same hour of the preceding day**:

$$\hat y_{D,h}=y_{D-1,h}.$$

On a complete hourly index, shifting the original series by 24 rows aligns those observations with their forecast targets. Select the test timestamps afterwards. All daily observations are known by the preceding day's 23:00 origin, including the 23:00 observation.

In [ ]:
# TODO: shift the original load_series by 24 hours and select test_index.
daily_predictions = None
raise NotImplementedError("Complete the daily seasonal-naïve forecast.")
forecasts["Seasonal naive (24 h)"] = daily_predictions

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
test_actual.iloc[:48].plot(ax=ax, color="black", label="Actual")
forecasts["Seasonal naive (24 h)"].iloc[:48].plot(ax=ax, label="Seasonal naïve (24 h)")
ax.set(title="Daily seasonal naïve: repeat the preceding day's profile", xlabel="Timestamp", ylabel="Demand (MW)")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Weekly seasonal-naïve forecast

### Task 3

Copy demand from the **same hour and weekday one week earlier**:

$$\hat y_{D,h}=y_{D-7,h}.$$

Use a shift of 168 hours. Later test forecasts can use earlier **observed** test demand: by their own daily origins, those observations are already available. Neither seasonal benchmark uses future observations or previous forecasts as donors.

In [ ]:
# TODO: shift the original load_series by 168 hours and select test_index.
weekly_predictions = None
raise NotImplementedError("Complete the weekly seasonal-naïve forecast.")
forecasts["Seasonal naive (168 h)"] = weekly_predictions

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
test_actual.iloc[:48].plot(ax=ax, color="black", label="Actual")
forecasts["Seasonal naive (168 h)"].iloc[:48].plot(ax=ax, label="Seasonal naïve (168 h)")
ax.set(title="Weekly seasonal naïve: repeat the preceding week's matching day", xlabel="Timestamp", ylabel="Demand (MW)")
ax.legend()
fig.tight_layout()
plt.show()

## 6. Linear regression

### Task 4a: explore potential predictors

A linear model combines predictors instead of copying one historical observation. Before choosing its inputs, examine the **training** demand only.

Start with the autocorrelation plot to find important lags. Then compare current demand $y_t$ with $y_{t-1}$, $y_{t-24}$ and $y_{t-168}$ in the scatter plots. Which relationship is strongest? What do the daily and weekly peaks suggest about potential predictors?

These plots are exploratory. A useful predictor must also be available when we issue the forecast.

### Autocorrelation and lag relationships

The supplied plotting function shows the first 300 hourly lags. Values close to 1 indicate a strong positive relationship. Examine lag 1 and look for repeated peaks near multiples of 24 hours, including lag 168. The grey lines are approximate 95% (solid) and 99% (dashed) reference limits for an uncorrelated series. 

Use the scatter plots below to examine the relationships at the three selected lags more closely.

In [ ]:
# Supplied pandas plotting function: use training observations only.
fig, ax = plt.subplots(figsize=(12, 3.5))
pd.plotting.autocorrelation_plot(training_load, ax=ax, color="tab:blue",
                                marker="o", markersize=3, linewidth=0.8)
ax.set(title="Autocorrelation of training demand", xlabel="Lag (hours)",
       ylabel="Autocorrelation", xlim=(0, 300), ylim=(-1, 1))
fig.tight_layout()
plt.show()

In [ ]:
# Lag exploratory plot
lagged_load = pd.DataFrame(index=training_load.index)
lagged_load["current"] = training_load
for lag in [1, 24, 168]:
    lagged_load[f"lag_{lag}"] = training_load.shift(lag)
lagged_load = lagged_load.dropna()

fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True, sharey=True)
for ax, lag in zip(axes, [1, 24, 168]):
    ax.scatter(lagged_load["current"], lagged_load[f"lag_{lag}"],
               s=15, alpha=0.35, color="tab:blue")
    ax.set_ylabel(rf"$y_{{t-{lag}}}$ (MW)")
axes[0].set_title("Training demand and lagged observations")
axes[-1].set_xlabel(r"$y_t$ (MW)")
fig.tight_layout()
plt.show()

**Discuss before selecting predictors:** Why is $y_{t-1}$ unavailable as an observed predictor for most target hours in our day-ahead forecast, even if its relationship with $y_t$ is strong? What other predictors can be constructed from the ones that are available?

### Task 4b: select predictors and construct the model inputs

Use demand at $t-24$ and $t-168$, together with a weekend indicator, as the linear model's three inputs:

| Columns | Construction |
|---|---|
| load_lag_24, load_lag_168 | Demand 24 and 168 hours before the target |
| is_weekend | 1 for Saturday/Sunday, otherwise 0 |

Calendar variables refer to the **target** time and are known in advance. The load lags must be known at its **daily origin**. Construct lags on the complete hourly series before splitting, so test predictors retain their preceding training observations.

After constructing these columns, create X_train, y_train and X_test using the supplied split code. The first 168 rows have an unavailable weekly lag. Drop incomplete **training** rows only and retain all test targets.

In [ ]:
index = load_series.index
features = pd.DataFrame(index=index)
# TODO: add load_lag_24, load_lag_168 and is_weekend.
# Hint: use load_series.shift for lags and index.dayofweek >= 5 for weekends.
raise NotImplementedError("Complete the three load-lag and weekend predictors.")

# Visualise the features.
display(features.head(30))

In [ ]:
X_train = features.loc[TRAIN_START:TRAIN_END].dropna()
y_train = load_series.loc[X_train.index]
X_test = features.loc[TEST_START:TEST_END]

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(test_actual.index) and X_test.notna().all().all()

print(f"Usable training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Removed {len(features.loc[TRAIN_START:TRAIN_END]) - len(X_train)} initial training rows with missing lags.")
display(X_train.index[:5])

### Task 4c: fit once and forecast

Create a LinearRegression model, fit it on X_train/y_train, then predict X_test. No separate forecasting function or retraining loop is needed, as every row of X_test already contains the observations available at its daily origin.

The model has an intercept and estimates one coefficient per input. Keep its fitted coefficients fixed throughout testing. Project any negative demand forecasts to zero. This is the same fit–predict workflow used for assessment load regression; regularised regression adds a penalty but follows the same steps.

In [ ]:
# TODO: create LinearRegression, fit on X_train/y_train, then predict X_test.
linear_model = None
linear_predictions = None  # A Series indexed by test_index.
# TODO: project negative forecasts to zero and add the "Linear regression" column.
raise NotImplementedError("Complete linear model fitting and forecasts.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
test_actual.iloc[:48].plot(ax=ax, color="black", label="Actual")
forecasts["Linear regression"].iloc[:48].plot(ax=ax, label="Linear regression")
ax.set(title="Linear regression: combine available lags and calendar inputs", xlabel="Timestamp", ylabel="Demand (MW)")
ax.legend()
fig.tight_layout()
plt.show()

## 7. Forecast error metrics

### Task 5: implement three reusable functions

Use actual minus forecast:

$$e_t=y_t-\hat y_t.$$

$$\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_t e_t^2},\qquad
\mathrm{MAE}=\frac{1}{N}\sum_t |e_t|,\qquad
\mathrm{MAPE}=\frac{100}{N}\sum_t\left|\frac{e_t}{y_t}\right|.$$

RMSE and MAE are in MW; MAPE is a percentage. RMSE gives large errors more weight, while MAPE measures error relative to demand.

Each function takes two pandas Series. Check identical lengths and timestamps **before** converting them to NumPy arrays. MAPE requires strictly positive actual demand here: raise an error for zero/non-positive values rather than adding a constant to its denominator.

In [ ]:
def rmse(actual, forecast):
    """Return root mean squared error in MW. Both inputs are aligned pandas Series."""
    # TODO: check lengths and timestamps, then apply the definition.
    raise NotImplementedError("Complete rmse.")

In [ ]:
def mae(actual, forecast):
    """Return mean absolute error in MW. Both inputs are aligned pandas Series."""
    # TODO: check lengths and timestamps, then apply the definition.
    raise NotImplementedError("Complete mae.")

In [ ]:
def mape(actual, forecast):
    """Return mean absolute percentage error (%). Both inputs are aligned pandas Series."""
    # TODO: check lengths and timestamps, then apply the definition.
    # TODO: reject zero/non-positive actual values explicitly.
    raise NotImplementedError("Complete mape.")

## 8. Compare all four methods

### Task 6: create the comparison table

Score every method on the **same complete test set**, not just the displayed week. Put models in rows and RMSE, MAE and MAPE in columns. Do not tune the model or choose predictors using these test scores.

In [ ]:
assert forecasts.index.equals(test_actual.index)
rows = []
# TODO: loop over the model columns, calculate all three metrics and create comparison.
# Columns: RMSE (MW), MAE (MW), MAPE (%); model names form the index.
comparison = None
raise NotImplementedError("Complete the full-test comparison table.")

## 9. Visualise forecasts and errors

### Forecasts for one week

Compare all four methods with actual demand over one test week. The comparison table above uses the full test set.

In [ ]:
# Supplied plot: one readable week from the full test set.
fig, ax = plt.subplots(figsize=(13, 4))
test_actual.loc[PLOT_START:PLOT_END].plot(ax=ax, color="black", linewidth=2, label="Actual")
forecasts.loc[PLOT_START:PLOT_END].plot(ax=ax, linewidth=1)
ax.set(title="Daily 24-hour forecasts: one week of the test set",
       xlabel="Target timestamp", ylabel="Demand (MW)")
ax.legend(ncol=3, fontsize=9)
fig.tight_layout()
plt.show()

### Error distributions

Compare the signed errors for linear regression and daily seasonal naïve. Positive errors mean the forecast was below actual demand; negative errors mean it was above.

In [ ]:
# Visualise some errors (actual minus forecast).

linear_errors = test_actual - forecasts["Linear regression"]
print(forecasts.head())
for method in ['Linear regression', 'Seasonal naive (24 h)']:
    errors = test_actual - forecasts[method]
    print(f"{method}: mean error = {errors.mean():.3f}, std error = {errors.std():.3f}")
    plt.hist(errors, bins=50, alpha=0.7, label = method)
    plt.xlabel("Prediction Error (MW)")
    plt.ylabel("Frequency")
    plt.title("Distribution of Forecast Errors")
plt.legend()
plt.show()

### Errors by forecast hour

The supplied plot groups test targets by hour of day. Each score uses 92 observations—one from every test day. At the 23:00 origin, hour 0 has lead time 1 and hour 23 has lead time 24. This experiment cannot separate time-of-day effects from forecast-lead effects.

Look for hours where each method struggles, and compare RMSE and MAE: both are in MW, but RMSE gives large errors more weight. In the assessment, you can apply the same grouping idea within separate weekly blocks.

In [ ]:
hourly_rmse = pd.DataFrame(index=range(24), columns=forecasts.columns, dtype=float)
hourly_mae = hourly_rmse.copy()
for hour in range(24):
    selected = test_actual.index.hour == hour
    for model in forecasts:
        hourly_rmse.loc[hour, model] = rmse(test_actual.loc[selected], forecasts.loc[selected, model])
        hourly_mae.loc[hour, model] = mae(test_actual.loc[selected], forecasts.loc[selected, model])

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
hourly_rmse.plot(ax=axes[0], marker="o", markersize=4)
hourly_mae.plot(ax=axes[1], marker="o", markersize=4, legend=False)
axes[0].set(title="Full-test errors by target hour", ylabel="RMSE (MW)")
axes[0].legend(ncol=2, fontsize=9)
axes[1].set(xlabel="Target hour of day (lead time = hour + 1)", ylabel="MAE (MW)")
axes[1].set_xticks(range(0, 24, 2))
fig.tight_layout()
plt.show()

## 10. Discuss the results

1. Which model has the lowest RMSE, MAE and MAPE? Are the rankings identical?
2. Why is daily persistence flat, while seasonal naïve and linear regression vary within the forecast day?
3. Why is the linear regression model very similar to seasonal naive (24 h)? Which parameters do you need to check?
4. When might the daily benchmark work better than the weekly benchmark, and vice versa?
5. Why is using an earlier observed test value as a later day's lag not data leakage? Would using the preceding test hour be valid for every target in this day-ahead experiment?
6. What do the lag scatter plots and calendar features contribute to the linear model? Which relevant influences are still missing?
7. At which hours are errors largest? Why might RMSE and MAE emphasise different hours?

**Link to the assessment.** The imputation tutorial practises data-quality checks, preserving observed values and comparing reconstructions. This notebook practises available predictors, chronological splitting, fitting, predicting, metrics and grouped errors. Keep these steps visible in your own work; complex helper functions are not required.

## 11. Optional extension: one-step-ahead load forecasting

Use the same load series, training/test dates and four methods, but move the forecast origin forward **one hour** instead of one day. At origin $t$, observe $y_t$ and forecast $y_{t+1}$. Once that next observation arrives, move forward and repeat.

Keep the tables indexed by the **target timestamp**, as above. For a target labelled $t$, the current observation at its origin is $y_{t-1}$; use `shift(1)` to align it. Persistence now follows the latest hourly observation, and regression can use lag 1 as well as lags 24/168 and the weekend flag. The seasonal-naïve rules do not change.

Fit regression once on the same training period and keep its coefficients fixed. During testing, update lagged **observations** at each hourly origin; do not use a forecast as the next observed value. This is a sequence of one-hour-ahead forecasts, not a recursive forecast of a full day from one origin. No further exploratory plots are needed.

### Create the one-step datasets

Copy the earlier predictors and add `load_lag_1`. Construct it on the complete series before selecting training and test rows. Keep every test target, so the two forecast settings are scored on exactly the same observations.

In [ ]:
one_step_features = features.copy()
# TODO: add load_lag_1 using the complete load series and shift(1).
raise NotImplementedError("Add the observation available at each hourly origin.")
ONE_STEP_COLUMNS = ["load_lag_1", "load_lag_24", "load_lag_168", "is_weekend"]

X_train_one_step = one_step_features.loc[TRAIN_START:TRAIN_END, ONE_STEP_COLUMNS].dropna()
y_train_one_step = load_series.loc[X_train_one_step.index]
X_test_one_step = one_step_features.loc[TEST_START:TEST_END, ONE_STEP_COLUMNS]
one_step_origins = pd.Series(test_index - pd.Timedelta(hours=1), index=test_index)
one_step_forecasts = pd.DataFrame(index=test_index)

assert X_train_one_step.index.equals(y_train_one_step.index)
assert X_train_one_step.index.equals(X_train.index)
assert X_test_one_step.index.equals(test_actual.index) and X_test_one_step.notna().all().all()
assert (one_step_origins.to_numpy() < test_index.to_numpy()).all()
print(f"One-step training rows: {len(X_train_one_step):,}; test rows: {len(X_test_one_step):,}.")
print(f"First two hourly origins: {one_step_origins.iloc[0]}, {one_step_origins.iloc[1]}.")

### Persistence

At each hourly origin, use the current observation as the next hour's forecast: $\hat y_{t+1\mid t}=y_t$. With target-indexed rows, this is the available lag-1 column.

In [ ]:
# TODO: store the available lag-1 observation as the persistence forecast.
raise NotImplementedError("Complete one-step persistence.")

### Daily seasonal naïve

Copy the target's same-hour observation from 24 hours earlier, just as in the day-ahead experiment.

In [ ]:
# TODO: store the available 24-hour lag as the daily seasonal-naïve forecast.
raise NotImplementedError("Complete the one-step daily seasonal benchmark.")

### Weekly seasonal naïve

Copy the target's same hour and weekday from 168 hours earlier. Its forecasts are also unchanged.

In [ ]:
# TODO: store the available 168-hour lag as the weekly seasonal-naïve forecast.
raise NotImplementedError("Complete the one-step weekly seasonal benchmark.")

### Linear regression

Fit a separate LinearRegression model on the four one-step predictors. The extra lag-1 input is available at every hourly origin. Predict the full test table and project negative demand forecasts to zero, as before.

In [ ]:
one_step_linear_model = LinearRegression()
# TODO: fit on X_train_one_step/y_train_one_step and predict X_test_one_step.
one_step_linear_predictions = None  # Return a Series indexed by test_index.
raise NotImplementedError("Complete the one-step linear fit and forecast.")
one_step_forecasts["Linear regression"] = one_step_linear_predictions.clip(lower=0)

### Estimate errors on the full test set

Reuse `rmse`, `mae` and `mape`. Score all four methods using actual demand first and forecasts second. The side-by-side table compares both settings on the same 2,208 test hours.

In [ ]:
assert one_step_forecasts.index.equals(test_actual.index)
assert np.isfinite(one_step_forecasts.to_numpy()).all()
one_step_rows = []
# TODO: score each method with RMSE, MAE and MAPE and build one_step_comparison.
one_step_comparison = None
raise NotImplementedError("Complete the full-test one-step error table.")
display(one_step_comparison.round(3))

horizon_comparison = pd.concat({"Day-ahead": comparison, "One-step ahead": one_step_comparison}, axis=1)
display(horizon_comparison.round(3))

### Visualise performance

Use the same test week for both settings, with matching axes. The metrics above use the full test period, not only this week.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, sharey=True)
for ax, title, prediction_table in [
    (axes[0], "Day-ahead: lead times 1–24 hours", forecasts),
    (axes[1], "One-step ahead: lead time always 1 hour", one_step_forecasts),
]:
    test_actual.loc[PLOT_START:PLOT_END].plot(ax=ax, color="black", linewidth=2, label="Actual")
    prediction_table.loc[PLOT_START:PLOT_END].plot(ax=ax, linewidth=1)
    ax.set(title=title, ylabel="Demand (MW)")
    ax.legend(ncol=3, fontsize=9)
axes[0].set_xlabel("")
axes[1].set_xlabel("Target timestamp")
fig.tight_layout()
plt.show()

**Compare the forecast horizons:** Using the table and plots, how do persistence and linear regression change from day-ahead to one-step-ahead forecasting, and why do the seasonal-naïve scores stay the same? Explain why observed lag 1 is valid here but unavailable for most targets in a full day-ahead forecast.